# 34. Scratch 정보보존 실험

scratch가 patch embedding과 stage 축소 과정에서 실제로 불리한 token coverage를 갖는지 확인합니다.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "ch3_utils.py").exists():
    matches = list(Path.cwd().glob("Deeplearning/*/3장/ch3_utils.py")) + list(Path.cwd().glob("**/ch3_utils.py"))
    if matches:
        NOTEBOOK_DIR = matches[0].parent
    else:
        NOTEBOOK_DIR = Path("Deeplearning") / "Vision 응용" / "3장"
sys.path.insert(0, str(NOTEBOOK_DIR))

from ch3_utils import *

paths = find_ch3_paths()
set_korean_font()
set_seed(31)
paths

Chapter3Paths(chapter3_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장'), chapter2_2_dir=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장'), data_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data'), stress_ladder_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/data/synthetic_metal_stress_ladder'), runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs'), manifest_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/3장/runs/manifests'), ch2_2_runs_root=WindowsPath('C:/Users/준승/Desktop/2026-1/Study/Deeplearning/Vision 응용/2-2장/runs'))

## 34-1. token coverage 계산

In [2]:
samples = load_ch3_base_samples()
manifests = create_ch3_probe_manifests(samples, max_per_cell=None, seed=31)
out_dir = paths.runs_root / "scratch_retention"

coverage = compute_scratch_token_coverage(manifests["eval_matched"], out_dir)
display(coverage.head())
display(
    coverage.groupby(["stride", "defect_type"])[["coverage_ratio", "pixels_per_covered_token"]]
    .mean()
    .reset_index()
)

,sample_id,color_group,shape_group,defect_type,defect_area,defect_width,defect_length,stride,token_grid_h,token_grid_w,covered_tokens,total_tokens,coverage_ratio,pixels_per_covered_token
0,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,4,32,32,20,1024,0.019531,6.400
1,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,8,16,16,8,256,0.031250,16.000
2,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,16,8,8,4,64,0.062500,32.000
3,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,32,4,4,2,16,0.125000,64.000
4,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,165,3.443,58.045,4,32,32,24,1024,0.023438,6.875


,stride,defect_type,coverage_ratio,pixels_per_covered_token
0,4,dent,0.038037,11.772700
1,4,impact,0.012858,9.496279
2,4,scratch,0.022884,7.241509
3,4,stain,0.052393,12.349655
4,8,dent,0.052409,34.146355
5,8,impact,0.022591,21.828022
6,8,scratch,0.040820,16.470624
7,8,stain,0.068620,37.814380
8,16,dent,0.087240,83.507917
9,16,impact,0.044792,49.470833


## 34-2. baseline 성능과 token coverage 병합

In [3]:
baseline_metrics = paths.ch2_2_runs_root / "baseline_seed_repeats" / "seed_0" / "sample_metrics.csv"
if not baseline_metrics.exists():
    raise FileNotFoundError("2-2장 baseline seed_0 sample_metrics.csv가 필요합니다.")

merged = merge_token_coverage_with_metrics(
    coverage,
    baseline_metrics,
    out_dir / "scratch_token_coverage_with_metrics.csv",
)
plot_token_coverage_relationship(merged, out_dir / "token_coverage_vs_dice.png")
display(merged.head())
display(
    merged[merged["stride"] == 4]
    .groupby("defect_type")[["coverage_ratio", "target_dice", "target_fnr"]]
    .mean()
    .reset_index()
    .sort_values("target_dice")
)

,sample_id,color_group,shape_group,defect_type,defect_area,defect_width,defect_length,stride,token_grid_h,token_grid_w,covered_tokens,total_tokens,coverage_ratio,pixels_per_covered_token,target_dice,target_fnr
0,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,4,32,32,20,1024,0.019531,6.400,0.714286,0.257812
1,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,8,16,16,8,256,0.031250,16.000,0.714286,0.257812
2,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,16,8,8,4,64,0.062500,32.000,0.714286,0.257812
3,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,128,2.805,36.444,32,4,4,2,16,0.125000,64.000,0.714286,0.257812
4,eval_matched_matched_control_neutral_scratch_t...,neutral,top_half_metal,scratch,165,3.443,58.045,4,32,32,24,1024,0.023438,6.875,0.175953,0.818182


,defect_type,coverage_ratio,target_dice,target_fnr
2,scratch,0.022884,0.283397,0.740849
3,stain,0.052393,0.521172,0.506109
1,impact,0.012858,0.532630,0.475027
0,dent,0.038037,0.615273,0.407527


## 34-3. stage energy retention 확인

In [4]:
feature_energy = paths.runs_root / "feature_bank" / "baseline_no_aug" / "seed_0" / "stage_energy.csv"
if feature_energy.exists():
    energy = pd.read_csv(feature_energy)
    idx = pd.read_csv(paths.runs_root / "feature_bank" / "baseline_no_aug" / "seed_0" / "sample_index.csv")
    energy = energy.merge(idx[["sample_id", "defect_type", "target_dice", "target_fnr"]], on="sample_id", how="left")
    energy.to_csv(out_dir / "scratch_retention_by_stage.csv", index=False, encoding="utf-8-sig")
    display(
        energy.groupby(["stage", "defect_type"])[["target_background_energy_ratio", "target_dice"]]
        .mean()
        .reset_index()
    )
else:
    print("stage_energy.csv가 없습니다. 31번 feature bank 추출을 먼저 실행하세요.")

,stage,defect_type,target_background_energy_ratio,target_dice
0,1,dent,1.055355,0.615273
1,1,impact,1.303376,0.532630
2,1,scratch,1.058238,0.283397
3,1,stain,0.767469,0.521172
4,2,dent,1.932761,0.615273
5,2,impact,1.907811,0.532630
6,2,scratch,1.711256,0.283397
7,2,stain,1.409720,0.521172
8,3,dent,1.312614,0.615273
9,3,impact,1.434683,0.532630
